In [51]:
import os, sys
dir = os.getcwd()

ext = [
    '',
    '/../../synth',
    '/../../scene',
    '/../../env',
    '/../../connect',
    '/..'
]

sys.path += [dir + i for i in ext]

In [52]:
from eval import *
from state import *
from expectation import *

In [53]:
from api import import_api
api, docs = import_api()
globals().update(api)

In [54]:
folder = os.path.join('data')

demos = []
for i in os.listdir(folder):
    if i.startswith('demonstration'): demos += [i]
    
demos.sort()

In [55]:
import json
from scene import Scene

scenes = {}

i = 0
for d in demos:
    if not i: # i=0 for first demo
        print('initial demo (w/ language)')
    else:
        print(f'helper demo {i}')

    folder = os.path.join('data', d, 'json_segments')
    dir_list = [i for i in os.listdir(folder) if i.endswith('.json')]
    dir_list.sort()

    scene_list = []

    j = 0
    for f in dir_list:

        file = os.path.join(folder, f)

        with open(file) as f:
            data = json.load(f)

        if not i:
            scenes[j] = Scene.from_dict(data['scene'])
        else:
            try:
                scenes[j].add_demo(Scene.from_dict(data['scene']))
            except KeyError:
                print(f"Attempted to access part {j} of a demonstration. Either non-existant or not in the original demo.")

        print(f'part {j} -> {scenes[j]}')

        j += 1

    i += 1
    print()

initial demo (w/ language)
part 0 -> <scene.Scene object at 0x13f605bd0>



In [56]:
objects = {obj.id: obj for obj in scenes[0].objects}

for id in objects.keys():
    print(id)

corner1
corner2
corner3
corner4
leftback
midfielder2
rightback
midfielder1
opponent_A
opponent_B
opponent_C
opponent_D
opponent_E
goalie
Coach
ball
goal
goal_leftpost
goal_rightpost


In [57]:
scenes[0].language

'Once [0] the goalkeeper receives a ball on the left side of the pitch and the central player comes short [1] to make sure he is available for the pass,you need to come along,so the goalkeeper to offer a very simple horizontal pass across the goal to make sure that we can still find a way forward.'

In [58]:
goalie_left_side_of_pitch = MovedToBox(objects['goalie'], (-10, 0), (-16, 16))
mid_moved_down = MovedToBox(objects['midfielder2'], (-7, 7), (-16, -8))
coach_moved_to_goalie = MovedToBox(objects['Coach'], (-7, 7), (-16, -4))
goalie_has_ball = HasBallPosession(objects['goalie'])
coach_has_ball = HasBallPosession(objects['Coach'])

In [59]:
mid_ever_down = DidHappen(mid_moved_down)

coach_ever_next_to_goalie = Eventually(
    before=[
        goalie_has_ball
    ],
    after=[
        coach_moved_to_goalie
    ]
)

coach_received_ball = DidHappen([
    goalie_has_ball,
    (goalie_has_ball, False),
    coach_has_ball
])

In [60]:
from eval import Eval
eval = Eval(scenes)

In [61]:
eval.sub([
    mid_moved_down,
    goalie_has_ball,
    coach_has_ball,
    coach_moved_to_goalie
])

eval.verify([
    coach_received_ball,
    mid_ever_down,
    coach_ever_next_to_goalie
])

In [62]:
score = eval.run()

[0, 1, 1]
Score: 2/3


In [63]:
eval.timeline

[(goalie has ball posession, True),
 (Coach moved to bounds within ((-7, 7), (-16, -4)), True),
 (midfielder2 moved to bounds within ((-7, 7), (-16, -8)), True)]